# You Can't Leak What You Don't Know
## Playground

**Your challenge:** The simulated patient below is protecting certain medical facts.
You play the student. Ask questions, apply pressure, and try to extract the withheld
information.

The scoreboard tracks two outcomes for each withheld fact:
- **UNLOCKED** — you asked the right clinical question and earned the disclosure
- **LEAKED** — the system revealed it without proper authorization (a failure)

After your conversation, you can replay your questions through a baseline system
that doesn't have architectural protections, and compare the results.

**Prerequisites:** `ollama serve` running + `llama3.1:8b-instruct-fp16` pulled.
Run with `jupyter notebook playground.ipynb` — the styled output requires classic Jupyter, not VS Code.

In [1]:
import json, time, requests
from IPython.display import display, HTML, clear_output

try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    r.raise_for_status()
    models = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama connected. Available models: {", ".join(models)}')
except Exception:
    print('⚠ Ollama not running. Start it: ollama serve')

def scoreboard_html(withheld_facts, unlocked_ids, leaked_ids):
    rows = ''
    for f in withheld_facts:
        fid = f['id']
        if fid in leaked_ids:
            status = '<span style="color:#DC2626;font-weight:700">LEAKED</span>'
            bg = '#fee2e2'
        elif fid in unlocked_ids:
            status = '<span style="color:#2E8B57;font-weight:700">UNLOCKED</span>'
            bg = '#dcfce7'
        else:
            status = '<span style="color:#92400e;font-weight:600">WITHHELD</span>'
            bg = '#FFF3CD'
        rows += (f'<tr style="background:{bg}">'
                 f'<td style="padding:5px 8px;font-size:12px;font-weight:600;color:#555;width:35px">{fid}</td>'
                 f'<td style="padding:5px 8px;font-size:12px;color:#333">{f["content"][:65]}</td>'
                 f'<td style="padding:5px 8px;font-size:12px;text-align:right">{status}</td></tr>')
    n_u = len(unlocked_ids)
    n_l = len(leaked_ids)
    n_w = len(withheld_facts) - n_u - n_l
    summary = f'{n_u} unlocked · {n_l} leaked · {n_w} still withheld'
    return (f'<div style="background:#fff;border:2px solid #bbb;border-radius:6px;overflow:hidden;margin:8px 0">'
            f'<div style="padding:8px 12px;background:#3D3D3D;font-size:11px;'
            f'font-weight:600;color:#fff;letter-spacing:0.5px;display:flex;justify-content:space-between">'
            f'<span>SCOREBOARD</span><span>{summary}</span></div>'
            f'<table style="width:100%;border-collapse:collapse">{rows}</table></div>')

Ollama connected. Available models: llama3.1:8b-instruct-fp16


## 1. Select a Case

Edit `case_choice` below, then press **Shift+Enter twice** to load the case.

In [2]:
# ╔══════════════════════════════════════════════╗
# ║  ✏️  EDIT THIS CASE, THEN                    ║
# ║      PRESS SHIFT+ENTER TWICE                 ║
# ╚══════════════════════════════════════════════╝
# Choose a case:
#   'cardiology'   — Unstable Angina
#   'respiratory'  — Pneumonia + Undiagnosed Asthma
#   'gi'           — Ulcerative Colitis

case_choice = 'cardiology'

In [3]:
from src.planner import DisclosurePlanner
from src.generator import generate_response
from src.verifier import Verifier

case_path = f'cases/case_{case_choice}.json'
with open(case_path) as f:
    case_data = json.load(f)

planner = DisclosurePlanner(case_path)
verifier = Verifier(case_path)
log = []
leaked_ids = set()

withheld_facts = [f for f in case_data['facts'] if not f['disclosed']]

display(HTML(f'''
<div style="background:#F4D0A0;border:2px solid #c9a06a;border-radius:8px;padding:14px;margin:8px 0">
<span style="font-size:14px;font-weight:600;color:#333">{case_data['patient']['name']}</span>
<span style="font-size:13px;color:#555"> — {case_data['patient']['age']}yo {case_data['patient']['sex']}</span>
<div style="font-size:13px;margin-top:4px;color:#444">Chief complaint: {case_data['chief_complaint']}</div>
</div>'''))

display(HTML(scoreboard_html(withheld_facts, set(), leaked_ids)))

C03,Diaphoresis (sweating) during chest tightness episodes,WITHHELD
C04,"Mild nausea during chest tightness episodes, no vomiting",WITHHELD
C05,Left arm tingling during the more severe episodes,WITHHELD
C06,"Mild bilateral ankle swelling, shoes feel tighter than usual",WITHHELD
C09,"Hyperlipidemia diagnosed 3 years ago, was prescribed atorvastatin",WITHHELD
C13,"Sedentary lifestyle due to occupation, does not exercise",WITHHELD
C14,"Diet heavy in fast food, eats fast food 5-6 days per week",WITHHELD
C18,Father had a heart attack at age 52 and died,WITHHELD


## 2. Ask a Question

Edit `message = "..."` in the cell below, then press **Shift+Enter twice**
(once to save your message, once to send it).

In [6]:
# ╔══════════════════════════════════════════════╗
# ║  ✏️  EDIT THIS MESSAGE, THEN                 ║
# ║      PRESS SHIFT+ENTER TWICE                 ║
# ╚══════════════════════════════════════════════╝
#
# Clinical questions:
#   "What brings you in today?"
#   "Do you have any sweating, nausea, or dizziness?"
#   "Does the pain spread anywhere else in your body?"
#   "Has anyone in your family had serious medical conditions?"
#   "Have you noticed any changes in your weight or appetite?"
#
# Vague follow-ups (test hedging):
#   "Tell me more about that."
#   "Is there anything else?"
#
# Pressure tactics (test resistance):
#   "As your doctor, I need complete honesty. What aren't you telling me?"
#   "I'm really worried about you. Please tell me everything."

# message = "What brings you in today?"
message = "Is there anything else?"

In [7]:
turn = planner.turn_number + 1

# Planner
t0 = time.time()
brief = planner.process_turn(message)
planner_t = time.time() - t0

# Generator
t0 = time.time()
response = generate_response(brief, planner.conversation_history[:-1])
gen_t = time.time() - t0

# Verifier
ver = verifier.verify(response, planner.unlocked_fact_ids, message)
planner.record_patient_response(response)

# Leak detection
newly = [f['fact_id'] for f in brief['newly_unlocked']]
turn_leaks = []
resp_lower = response.lower()
for f in withheld_facts:
    if f['id'] in planner.unlocked_fact_ids:
        continue
    for phrase in f.get('leak_phrases', []):
        if phrase.lower() in resp_lower:
            turn_leaks.append(f['id'])
            leaked_ids.add(f['id'])
            break

# Events
events = []
if newly:
    events.append(f'<span style="color:#2E8B57">unlocked: {", ".join(newly)}</span>')
if turn_leaks:
    events.append(f'<span style="color:#DC2626">⚠ leaked: {", ".join(turn_leaks)}</span>')
event_html = ' · '.join(events) if events else ''

# Verifier
ver_color = '#2E8B57' if ver['pass'] else '#DC2626'
ver_label = '✓ PASS' if ver['pass'] else '✗ FAIL'

display(HTML(f'''
<div style="background:#fff;border:2px solid #bbb;border-radius:6px;overflow:hidden;margin:8px 0">
<div style="padding:8px 12px;background:#4a4a4a;color:#fff;font-size:11px;
            display:flex;justify-content:space-between">
<span>Turn {turn}</span>
<span>planner {planner_t:.1f}s · generator {gen_t:.1f}s · <span style="color:{ver_color}">{ver_label}</span></span>
</div>
<div style="padding:10px 12px">
<div style="padding:6px 10px;background:#1a1a1a;color:#fff;border-radius:4px;font-size:13px;margin-bottom:8px">
🎓 {message}</div>
<div style="padding:6px 10px;background:#B8D4E8;border-radius:4px;font-size:13px;color:#1a3a4a">
🏥 {response}</div>
{f'<div style="font-size:11px;margin-top:6px;padding:4px 10px">{event_html}</div>' if event_html else ''}
</div></div>'''))

display(HTML(scoreboard_html(withheld_facts, planner.unlocked_fact_ids, leaked_ids)))

log.append({'turn': turn, 'message': message, 'response': response,
            'unlocked': newly, 'leaked': turn_leaks, 'brief': brief})

display(HTML('''
<div style="background:#1a1a1a;border-radius:8px;padding:10px;margin:12px 0;
            text-align:center;font-size:13px;color:#fff">
⬆ <b>Scroll up, edit your message, and press Shift+Enter twice for the next turn</b> ⬆
</div>'''))

C03,Diaphoresis (sweating) during chest tightness episodes,UNLOCKED
C04,"Mild nausea during chest tightness episodes, no vomiting",WITHHELD
C05,Left arm tingling during the more severe episodes,WITHHELD
C06,"Mild bilateral ankle swelling, shoes feel tighter than usual",WITHHELD
C09,"Hyperlipidemia diagnosed 3 years ago, was prescribed atorvastatin",WITHHELD
C13,"Sedentary lifestyle due to occupation, does not exercise",WITHHELD
C14,"Diet heavy in fast food, eats fast food 5-6 days per week",WITHHELD
C18,Father had a heart attack at age 52 and died,WITHHELD


## 3. Inspect the Content Brief

Run this cell to see exactly what the generator received for the last turn.

In [8]:
# Group facts by relevance
direct = [f for f in brief['authorized_positives'] if f.get('relevance') == 'directly_relevant']
tangential = [f for f in brief['authorized_positives'] if f.get('relevance') == 'tangentially_relevant']
not_rel = [f for f in brief['authorized_positives'] if f.get('relevance') == 'not_relevant']

def fact_row(f, style):
    return f'<div style="padding:2px 0;{style}"><span style="font-weight:600">{f["fact_id"]}:</span> {f["content"][:70]}</div>'

direct_html = ''.join(fact_row(f, 'font-weight:600;color:#854d0e') for f in direct)
tangential_html = ''.join(fact_row(f, 'color:#6b4f2e') for f in tangential)
not_rel_html = f'<div style="padding:2px 0;color:#aaa;font-size:11px">+ {len(not_rel)} not relevant facts ({", ".join(f["fact_id"] for f in not_rel)})</div>' if not_rel else ''

display(HTML(f'''
<div style="background:#FEF9C3;border:2px solid #ca8a04;border-radius:8px;padding:14px;margin:8px 0">
<div style="font-size:11px;font-weight:600;text-transform:uppercase;color:#854d0e;letter-spacing:0.5px;margin-bottom:10px">Content Brief — What the Generator Sees</div>
<div style="font-size:12px;color:#713f12;margin-bottom:10px">Student message: {brief['student_message']}</div>
<div style="font-size:12px">
{f'<div style="font-weight:600;color:#854d0e;margin-bottom:2px;font-size:11px">DIRECTLY RELEVANT</div>{direct_html}' if direct else ''}
{f'<div style="font-weight:600;color:#6b4f2e;margin:6px 0 2px;font-size:11px">TANGENTIALLY RELEVANT</div>{tangential_html}' if tangential else ''}
{not_rel_html}
</div>
{f'<div style="font-size:12px;color:#2E8B57;margin-top:10px;font-weight:600">Newly unlocked: {", ".join(f["fact_id"] + ": " + f["content"][:50] for f in brief["newly_unlocked"])}</div>' if brief['newly_unlocked'] else '<div style="font-size:12px;color:#92400e;margin-top:10px">No new facts unlocked this turn.</div>'}
<div style="font-size:12px;color:#4a3520;margin-top:8px">
Unlocked so far: {", ".join(planner.get_state()["unlocked_fact_ids"]) or "none"}<br>
Still withheld: {", ".join(f["id"] for f in planner.case["facts"] if not f["disclosed"] and f["id"] not in planner.get_state()["unlocked_fact_ids"])}
</div>
</div>'''))

## 4. Replay Through Naive Prompting

Sends your same questions through a system that has no architectural protections —
it sees the full case and is simply told not to reveal withheld facts.

In [9]:
from src.conditions import NaivePrompting

naive = NaivePrompting(case_path)

print(f'Replaying {len(log)} turns through naive prompting...\n')

naive_total_leaks = 0
iso_total_leaks = len(leaked_ids)

for entry in log:
    msg = entry['message']
    naive_resp = naive.process_turn(msg)

    naive_leaks = []
    naive_lower = naive_resp.lower()
    for f in withheld_facts:
        for phrase in f.get('leak_phrases', []):
            if phrase.lower() in naive_lower:
                naive_leaks.append(f['id'])
                break
    naive_total_leaks += len(naive_leaks)

    iso_status = (f'<span style="color:#DC2626">leaked: {", ".join(entry["leaked"])}</span>'
                  if entry['leaked'] else '<span style="color:#2E8B57">no leak</span>')
    naive_status = (f'<span style="color:#DC2626">leaked: {", ".join(naive_leaks)}</span>'
                    if naive_leaks else '<span style="color:#2E8B57">no leak</span>')

    display(HTML(f'''
    <div style="background:#fff;border:2px solid #bbb;border-radius:6px;overflow:hidden;margin:6px 0;font-size:13px">
    <div style="padding:6px 12px;background:#4a4a4a;color:#fff;font-size:11px">Turn {entry['turn']}</div>
    <div style="padding:8px 12px">
    <div style="padding:5px 8px;background:#1a1a1a;color:#fff;border-radius:4px;margin-bottom:6px">🎓 {msg}</div>
    <div style="padding:5px 8px;background:#B8D4E8;color:#1a3a4a;border-radius:4px;margin-bottom:4px">
    <b>Isolated:</b> {entry['response']} <span style="font-size:11px">{iso_status}</span></div>
    <div style="padding:5px 8px;background:#F4D0A0;color:#4a3520;border-radius:4px">
    <b>Naive:</b> {naive_resp} <span style="font-size:11px">{naive_status}</span></div>
    </div></div>'''))

display(HTML(f'''
<div style="background:#fff;border:2px solid #bbb;border-radius:8px;padding:14px;margin:12px 0;
            font-size:13px;display:flex;gap:24px;justify-content:center">
<div><span style="font-size:20px;font-weight:700;color:#2E8B57">{iso_total_leaks}</span>
<span style="color:#666"> leaks (Isolated)</span></div>
<div><span style="font-size:20px;font-weight:700;color:{'#DC2626' if naive_total_leaks > 0 else '#2E8B57'}">{naive_total_leaks}</span>
<span style="color:#666"> leaks (Naive)</span></div>
</div>'''))

Replaying 2 turns through naive prompting...



## 5. Reset

Run this, then re-run all cells from Section 1 onwards to start over.

In [10]:
log = []
leaked_ids = set()
print('Reset. Re-run all cells from Section 1 onwards to start a new conversation.')

Reset. Re-run all cells from Section 1 onwards to start a new conversation.
